# Tanh Hidden Layer Multiclass Experiments

This notebook runs experiments for different values of `alpha` and `random_state`, plots decision boundaries and loss curves, and reports the best configuration.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

NUM_SAMPLES = 1280
NOISE = 0.2
TEST_SIZE = 0.2
EPOCHS = 25001
EPSILON = 1e-15

In [2]:
def fn_softmax(z: np.ndarray) -> np.ndarray:
    z_shifted = z - np.max(z, axis=1, keepdims=True)
    exp_sc = np.exp(z_shifted)
    return exp_sc / np.sum(exp_sc, axis=1, keepdims=True)

def fn_activ(z: np.ndarray) -> np.ndarray:
    return np.tanh(z)

def fn_activ_prime(z: np.ndarray) -> np.ndarray:
    return 1.0 - np.tanh(z) ** 2

In [3]:
def fn_calculate_loss(model: dict, X: np.ndarray, y: np.ndarray) -> float:
    W1, b1, W2, b2 = model['W1'], model['b1'], model['W2'], model['b2']
    m = X.shape[0]
    z1 = X.dot(W1) + b1
    a1 = fn_activ(z1)
    z2 = a1.dot(W2) + b2
    a2 = fn_softmax(z2)
    a2 = np.clip(a2, EPSILON, 1 - EPSILON)
    data_loss = -np.sum(y * np.log(a2))
    return data_loss / m

def fn_predict(model: dict, X: np.ndarray) -> np.ndarray:
    W1, b1, W2, b2 = model['W1'], model['b1'], model['W2'], model['b2']
    z1 = X.dot(W1) + b1
    a1 = fn_activ(z1)
    z2 = a1.dot(W2) + b2
    a2 = fn_softmax(z2)
    return np.argmax(a2, axis=1)

In [4]:
def fn_plot_decision_boundary(model: dict, X_tr: np.ndarray, y_tr: np.ndarray, X_ts: np.ndarray, y_ts: np.ndarray, title: str, filename: Path):
    x_min, x_max = X_tr[:, 0].min() - 0.05, X_tr[:, 0].max() + 0.05
    y_min, y_max = X_tr[:, 1].min() - 0.05, X_tr[:, 1].max() + 0.05
    h = 0.01
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = fn_predict(model, np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.contourf(xx, yy, Z, cmap='coolwarm', alpha=0.7)
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=np.argmax(y_tr, axis=1), s=30, edgecolor='k', cmap=plt.cm.coolwarm)
    ax.scatter(X_ts[:, 0], X_ts[:, 1], c=np.argmax(y_ts, axis=1), s=120, marker='*', edgecolor='k', cmap=plt.cm.inferno)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    fig.colorbar(ax.collections[0], ax=ax, label='Class')
    fig.tight_layout()
    fig.savefig(filename, dpi=150)
    plt.close(fig)

In [5]:
def fn_build_model(X: np.ndarray, y: np.ndarray, hidden_dim: int, alpha: float, epochs: int, rng: np.random.Generator) -> tuple[dict, list, list]:
    m, nn_input_dim = X.shape
    nn_output_dim = y.shape[1]
    W1 = rng.standard_normal((nn_input_dim, hidden_dim)) / np.sqrt(nn_input_dim)
    W2 = rng.standard_normal((hidden_dim, nn_output_dim)) / np.sqrt(hidden_dim)
    b1 = np.zeros((1, hidden_dim))
    b2 = np.zeros((1, nn_output_dim))

    loss_history = []
    epoch_history = []

    for i in range(epochs):
        z1 = X.dot(W1) + b1
        a1 = fn_activ(z1)
        z2 = a1.dot(W2) + b2
        a2 = fn_softmax(z2)

        dz2 = a2 - y
        dW2 = a1.T.dot(dz2)
        db2 = np.sum(dz2, axis=0, keepdims=True)

        da1 = dz2.dot(W2.T)
        dz1 = da1 * fn_activ_prime(z1)
        dW1 = X.T.dot(dz1)
        db1 = np.sum(dz1, axis=0, keepdims=True)

        W1 -= alpha * dW1 / m
        b1 -= alpha * db1 / m
        W2 -= alpha * dW2 / m
        b2 -= alpha * db2 / m

        if i % 100 == 0:
            model = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
            loss = fn_calculate_loss(model, X, y)
            loss_history.append(loss)
            epoch_history.append(i)

    model = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
    return model, epoch_history, loss_history

In [6]:
def run_experiments():
    alphas = [0.1, 0.01, 0.001]
    random_states = [24, 42, 0, 999]
    hidden_dim = 5

    summary = []

    for alpha in alphas:
        for random_state in random_states:
            print(f'Running experiment: alpha={alpha}, random_state={random_state}')
            rng = np.random.default_rng(seed=random_state)
            X, y = datasets.make_moons(n_samples=NUM_SAMPLES, shuffle=True, noise=NOISE, random_state=random_state)
            y = pd.get_dummies(y).to_numpy()
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=random_state)

            model, epochs, loss_history = fn_build_model(X_train, y_train, hidden_dim, alpha, EPOCHS, rng)

            y_test_pred = fn_predict(model, X_test)
            test_accuracy = accuracy_score(np.argmax(y_test, axis=1), y_test_pred)
            report = classification_report(np.argmax(y_test, axis=1), y_test_pred, output_dict=True)
            final_loss = loss_history[-1] if loss_history else fn_calculate_loss(model, X_train, y_train)

            experiment_name = f'alpha_{alpha}_rs_{random_state}'
            decision_path = OUTPUT_DIR / f'decision_{experiment_name}.png'
            loss_path = OUTPUT_DIR / f'loss_{experiment_name}.png'

            fn_plot_decision_boundary(model, X_train, y_train, X_test, y_test,
                                      title=f'alpha={alpha}, rs={random_state}, acc={test_accuracy:.4f}',
                                      filename=decision_path)

            fig, ax = plt.subplots(figsize=(8, 5))
            ax.plot(epochs, loss_history, marker='o', markersize=3)
            ax.set_title(f'Loss Curve alpha={alpha}, rs={random_state}')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.grid(True)
            fig.tight_layout()
            fig.savefig(loss_path, dpi=150)
            plt.close(fig)

            summary.append({
                'alpha': alpha,
                'random_state': random_state,
                'test_accuracy': test_accuracy,
                'final_loss': final_loss,
                'decision_plot': str(decision_path),
                'loss_plot': str(loss_path),
                'precision_macro': report['macro avg']['precision'],
                'recall_macro': report['macro avg']['recall'],
                'f1_macro': report['macro avg']['f1-score'],
            })

    summary_df = pd.DataFrame(summary)
    summary_path = OUTPUT_DIR / 'experiment_summary.csv'
    summary_df.to_csv(summary_path, index=False)

    best_accuracy = summary_df.loc[summary_df['test_accuracy'].idxmax()]
    print('Best configuration:')
    print(best_accuracy.to_string())
    print(f'Saved summary to: {summary_path}')

In [7]:
run_experiments()

Running experiment: alpha=0.1, random_state=24
Running experiment: alpha=0.1, random_state=42
Running experiment: alpha=0.1, random_state=0
Running experiment: alpha=0.1, random_state=999
Running experiment: alpha=0.01, random_state=24
Running experiment: alpha=0.01, random_state=42
Running experiment: alpha=0.01, random_state=0
Running experiment: alpha=0.01, random_state=999
Running experiment: alpha=0.001, random_state=24
Running experiment: alpha=0.001, random_state=42
Running experiment: alpha=0.001, random_state=0
Running experiment: alpha=0.001, random_state=999
Best configuration:
alpha                                             0.1
random_state                                        0
test_accuracy                                0.976562
final_loss                                   0.052736
decision_plot      output\decision_alpha_0.1_rs_0.png
loss_plot              output\loss_alpha_0.1_rs_0.png
precision_macro                              0.976439
recall_macro              